In [ ]:
# =============================================================================
# OPTIMIZACIÓN DE ABASTECIMIENTO — dataset_retail_unificado.csv
# Forecasting por SKU-Tienda (Quantile Regression) + Modelo "Newsvendor" (vendedor de periódicos para productos perecederos)
# =============================================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.stats import norm
from sklearn.metrics import mean_absolute_error, mean_squared_error
import lightgbm as lgb
import os

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURACIÓN INICIAL
# ─────────────────────────────────────────────────────────────────────────────
# Semanas para entrenamiento, validación y predicción
# Con 13 semanas históricas y lags de hasta 4 semanas:
#   - Semanas efectivas para entrenar: 5 → 11 (7 semanas)
#   - Validación (hold-out):           semana 12 y 13
#   - Forecast objetivo:               semana 14 (próxima semana real)
SEMANA_TEST_INICIO = 12   # nro de semana (1-indexed) donde empieza hold-out
LAGS              = [1, 2, 3, 4]
VENTANAS_ROLLING  = [2, 4]
QUANTILES         = {"P10": 0.10, "P50": 0.50, "P90": 0.90}

plt.rcParams.update({
    "figure.facecolor": "#F7F9FC",
    "axes.facecolor":   "#FFFFFF",
    "axes.grid":        True,
    "grid.alpha":       0.3,
    "font.family":      "DejaVu Sans",
    "axes.spines.top":  False,
    "axes.spines.right":False,
})

COLORES = {
    "azul_oscuro": "#1A5276",
    "azul_medio":  "#2980B9",
    "azul_claro":  "#AED6F1",
    "rojo":        "#E74C3C",
    "verde":       "#27AE60",
    "naranja":     "#E67E22",
    "gris":        "#7F8C8D",
}

print("=" * 70)
print("  OPTIMIZACIÓN DE ABASTECIMIENTO — dataset_retail_unificado.csv")
print("=" * 70)



In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECCIÓN 1 — CARGA Y VALIDACIÓN
# ─────────────────────────────────────────────────────────────────────────────
print("\n[1/7] CARGA Y VALIDACIÓN DEL DATASET")
print("-" * 50)

df = pd.read_csv("C:/Users/USUARIO/Documents/KONRAD LORENZ/MAESTRÍA/MLOPS/FORECAST/data/dataset_retail_unificado.csv")
df["fecha"] = pd.to_datetime(df["fecha"])
df = df.sort_values(["id_tienda", "id_producto", "fecha"]).reset_index(drop=True)

# Validación básica de integridad
assert df["unidades_vendidas"].min() >= 0,    "ERROR: unidades negativas detectadas"
assert df["precio_venta"].gt(df["costo_unitario"]).all(), "ERROR: precio < costo en algún producto"
assert df.isnull().sum().sum() == 0,          "ERROR: valores nulos detectados"

# Enriquecer con métricas de negocio
df["margen_unitario"]   = df["precio_venta"] - df["costo_unitario"]
df["margen_pct"]        = df["margen_unitario"] / df["precio_venta"]
df["semana_num"]        = (df["fecha"].dt.isocalendar().week - df["fecha"].dt.isocalendar().week.min() + 1).astype(int)
df["es_finde"]          = (df["dia_semana"] >= 4).astype(int)   # 4=Vie, 5=Sáb, 6=Dom

n_skus     = df.groupby(["id_tienda", "id_producto"]).ngroups
n_semanas  = df["fecha"].dt.to_period("W").nunique()
fecha_ini  = df["fecha"].min().date()
fecha_fin  = df["fecha"].max().date()

print(f"  Registros          : {len(df):,}")
print(f"  SKU-Tienda únicos  : {n_skus}")
print(f"  Tiendas            : {df['id_tienda'].nunique()}")
print(f"  Productos          : {df['id_producto'].nunique()}")
print(f"  Semanas históricas : {n_semanas}")
print(f"  Rango de fechas    : {fecha_ini} → {fecha_fin}")
print(f"  Nulos              : {df.isnull().sum().sum()}")
print(f"  Valores negativos  : {(df['unidades_vendidas'] < 0).sum()}")

# Catálogo para referencia rápida
catalogo = (
    df[["id_producto", "nombre", "categoria",
        "costo_unitario", "precio_venta",
        "costo_almacenamiento_semanal", "margen_unitario", "margen_pct"]]
    .drop_duplicates()
    .sort_values("id_producto")
    .reset_index(drop=True)
)
print(f"\n  Catálogo de productos:")
print(catalogo[["id_producto","nombre","costo_unitario","precio_venta",
                "margen_unitario","margen_pct"]].to_string(index=False))

# Stock actual (snapshot único por SKU-Tienda)
stock_df = (
    df[["id_tienda", "id_producto", "stock_actual"]]
    .drop_duplicates(subset=["id_tienda", "id_producto"])
    .reset_index(drop=True)
)



In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECCIÓN 2 — EDA
# ─────────────────────────────────────────────────────────────────────────────
OUTPUT_DIR = ('C:/Users/USUARIO/Documents/KONRAD LORENZ/MAESTRÍA/MLOPS/FORECAST/models')
print("\n[2/7] ANÁLISIS EXPLORATORIO DE DATOS (EDA)")
print("-" * 50)

# ── 2.1  Agregaciones clave ───────────────────────────────────────────────────
diario_total = df.groupby("fecha")["unidades_vendidas"].sum().reset_index()
diario_total["rolling7"] = diario_total["unidades_vendidas"].rolling(7, center=True, min_periods=1).mean()

por_producto = (
    df.groupby(["id_producto","nombre","categoria"])["unidades_vendidas"]
    .sum().reset_index().sort_values("unidades_vendidas", ascending=False)
)

patron_dow = df.groupby("dia_semana")["unidades_vendidas"].mean()
etiquetas_dow = ["Lun","Mar","Mié","Jue","Vie","Sáb","Dom"]

por_trend = df.groupby("trend_type")["unidades_vendidas"].mean()

pivot_sku = df.pivot_table(
    values="unidades_vendidas", index="nombre", columns="id_tienda", aggfunc="sum"
)

# Serie semanal por categoría
df["semana_periodo"] = df["fecha"].dt.to_period("W").apply(lambda r: r.start_time)
sem_cat = (
    df.groupby(["semana_periodo","categoria"])["unidades_vendidas"]
    .sum().reset_index()
)

# ── 2.2  Figura EDA (6 paneles) ───────────────────────────────────────────────
fig = plt.figure(figsize=(20, 16))
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.50, wspace=0.38)

# Panel A: Serie diaria total
ax_a = fig.add_subplot(gs[0, :])
ax_a.fill_between(diario_total["fecha"], diario_total["unidades_vendidas"],
                  alpha=0.18, color=COLORES["azul_medio"])
ax_a.plot(diario_total["fecha"], diario_total["unidades_vendidas"],
          color=COLORES["azul_claro"], lw=0.9, alpha=0.6)
ax_a.plot(diario_total["fecha"], diario_total["rolling7"],
          color=COLORES["azul_oscuro"], lw=2.3, label="Media móvil 7 días")

# Sombrear fines de semana
for group_name, grp in df.groupby("semana_periodo"):
    finde = df[(df["semana_periodo"] == group_name) & (df["es_finde"] == 1)]["fecha"]
    if len(finde):
        ax_a.axvspan(finde.min(), finde.max(), alpha=0.06, color=COLORES["naranja"])

patch_finde = mpatches.Patch(color=COLORES["naranja"], alpha=0.3, label="Fin de semana")
ax_a.legend(handles=[ax_a.lines[1], patch_finde], fontsize=10)
ax_a.set_title("Ventas diarias totales — 20 tiendas × 8 productos (91 días)",
               fontsize=13, fontweight="bold")
ax_a.set_ylabel("Unidades vendidas")
ax_a.set_xlabel("")

# Panel B: Consumo por producto
ax_b = fig.add_subplot(gs[1, :2])
colores_cat = [COLORES["azul_oscuro"] if c == "Bebidas" else COLORES["naranja"]
               for c in por_producto["categoria"]] # Fix here.
bars = ax_b.barh(por_producto["nombre"][::-1],
                 por_producto["unidades_vendidas"][::-1],
                 color=colores_cat[::-1], edgecolor="white")
ax_b.set_title("Volumen total por producto", fontsize=12, fontweight="bold")
ax_b.set_xlabel("Unidades vendidas")
for bar in bars:
    ax_b.text(bar.get_width() * 1.005, bar.get_y() + bar.get_height() / 2,
              f"{bar.get_width():,.0f}", va="center", fontsize=8.5)
leg_b = [mpatches.Patch(color=COLORES["azul_oscuro"], label="Bebidas"),
         mpatches.Patch(color=COLORES["naranja"], label="Alimentos")]
ax_b.legend(handles=leg_b, fontsize=9)

# Panel C: Patrón día de semana
ax_c = fig.add_subplot(gs[1, 2])
colores_dow = [COLORES["azul_oscuro"]]*4 + [COLORES["rojo"]]*3
bars_c = ax_c.bar(etiquetas_dow, patron_dow.values, color=colores_dow, edgecolor="white")
ax_c.set_title("Ventas promedio\npor día de semana", fontsize=12, fontweight="bold")
ax_c.set_ylabel("Unidades (promedio)")
incremento = (patron_dow.iloc[4:].mean() / patron_dow.iloc[:4].mean() - 1) * 100
ax_c.text(0.05, 0.92, f"Fines de semana\n+{incremento:.0f}% vs. L-J",
          transform=ax_c.transAxes, fontsize=9, color=COLORES["rojo"],
          bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor=COLORES["rojo"], alpha=0.8))

# Panel D: Serie semanal por categoría
ax_d = fig.add_subplot(gs[2, :2])
for cat, color in [("Bebidas", COLORES["azul_oscuro"]), ("Alimentos", COLORES["naranja"])]:
    sub = sem_cat[sem_cat["categoria"] == cat]
    ax_d.plot(sub["semana_periodo"], sub["unidades_vendidas"],
              marker="o", ms=5, lw=2, color=color, label=cat)
ax_d.set_title("Evolución semanal por categoría", fontsize=12, fontweight="bold")
ax_d.set_xlabel("Semana")
ax_d.set_ylabel("Unidades totales")
ax_d.legend()
ax_d.xaxis.set_tick_params(rotation=30)

# Panel E: Boxplot por trend_type
ax_e = fig.add_subplot(gs[2, 2])
trend_order = ["up", "seasonal", "random", "down"]
data_box    = [df[df["trend_type"] == t]["unidades_vendidas"].values for t in trend_order]
bp = ax_e.boxplot(data_box, labels=trend_order, patch_artist=True,
                  medianprops={"color": COLORES["rojo"], "linewidth": 2},
                  flierprops={"marker": ".", "color": COLORES["azul_claro"], "alpha": 0.4})
colores_bp = [COLORES["verde"], COLORES["azul_medio"], COLORES["gris"], COLORES["rojo"]]
for patch, color in zip(bp["boxes"], colores_bp):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
ax_e.set_title("Distribución por\ntipo de tendencia", fontsize=12, fontweight="bold")
ax_e.set_ylabel("Unidades vendidas")
ax_e.set_xlabel("trend_type")

fig.suptitle("EDA — Optimización de Abastecimiento Retail Colombia",
             fontsize=15, fontweight="bold", y=1.01)
plt.savefig(f"{OUTPUT_DIR}/01_eda_retail.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"  → 01_eda_retail.png guardado")

# ── 2.3 Heatmap SKU × Tienda ─────────────────────────────────────────────────
fig_h, ax_h = plt.subplots(figsize=(18, 6))
pivot_norm = pivot_sku.div(pivot_sku.max(axis=1), axis=0)   # normalizar por producto
sns.heatmap(pivot_norm, ax=ax_h, cmap="Blues", linewidths=0.3, linecolor="#EEEEEE",
            vmin=0, vmax=1, cbar_kws={"label": "Volumen relativo (max=1)"})
ax_h.set_title("Heatmap de Volumen Relativo por SKU × Tienda",
               fontsize=13, fontweight="bold")
ax_h.set_xlabel("Tienda")
ax_h.set_ylabel("Producto")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/02_heatmap_sku_tienda.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"  → 02_heatmap_sku_tienda.png guardado")

# Estadísticos clave para el log
print(f"\n  Hallazgos EDA:")
print(f"    Incremento fin de semana      : +{incremento:.0f}% vs. Lun-Jue")
print(f"    Producto más vendido           : {por_producto.iloc[0]['nombre']} ({por_producto.iloc[0]['unidades_vendidas']:,} u)")
print(f"    Producto menos vendido         : {por_producto.iloc[-1]['nombre']} ({por_producto.iloc[-1]['unidades_vendidas']:,} u)")
print(f"    Trend types en el dataset      : {df['trend_type'].value_counts().to_dict()}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECCIÓN 3 — FEATURE ENGINEERING SEMANAL
# ─────────────────────────────────────────────────────────────────────────────
print("\n[3/7] FEATURE ENGINEERING SEMANAL")
print("-" * 50)

# Agregación semanal
df["semana"] = df["fecha"].dt.to_period("W").apply(lambda r: r.start_time)
sem = (
    df.groupby(["semana", "id_tienda", "id_producto"])
    .agg(
        demanda_semanal    = ("unidades_vendidas",                  "sum"),
        dias_con_venta     = ("unidades_vendidas",                  lambda x: (x > 0).sum()),
        proporcion_finde   = ("es_finde",                           "mean"),
    )
    .reset_index()
)

# Añadir atributos de negocio (variables en el tiempo)
attrs = df[["id_tienda","id_producto","ciudad","tamaño_m2","nombre","categoria",
            "costo_unitario","precio_venta","costo_almacenamiento_semanal",
            "margen_unitario","margen_pct","trend_type"]].drop_duplicates()
sem = sem.merge(attrs, on=["id_tienda","id_producto"])

# Número de semana relativo (1-indexed)
semanas_ord = sorted(sem["semana"].unique())
semana_a_num = {s: i+1 for i, s in enumerate(semanas_ord)}
sem["semana_num"] = sem["semana"].map(semana_a_num)

sem = sem.sort_values(["id_tienda","id_producto","semana_num"]).reset_index(drop=True)
grp_sku = sem.groupby(["id_tienda","id_producto"])["demanda_semanal"]

# Lags semanales (retraso)
for lag in LAGS:
    sem[f"lag_{lag}w"] = grp_sku.shift(lag)

# Rolling sobre la serie shifteada 1 período (evita data leakage)
for v in VENTANAS_ROLLING:
    sem[f"rolling_mean_{v}w"] = grp_sku.shift(1).transform(
        lambda x: x.rolling(v, min_periods=1).mean()
    )
    sem[f"rolling_std_{v}w"] = grp_sku.shift(1).transform(
        lambda x: x.rolling(v, min_periods=1).std().fillna(0)
    )

# Coeficiente de variación
sem["cv"] = sem["rolling_std_4w"] / (sem["rolling_mean_4w"] + 1e-9)

# Ratio tienda vs. media del producto (poder relativo de la tienda)
media_prod_sem = sem.groupby(["id_producto","semana_num"])["demanda_semanal"].transform("mean")
sem["ratio_tienda_vs_media"] = sem["demanda_semanal"] / (media_prod_sem + 1e-9)

# Tendencia semana a semana (diferencia de lag)
sem["delta_lag1_lag2"] = sem["lag_1w"] - sem["lag_2w"]

# Encodings categóricos
sem["trend_enc"]    = pd.Categorical(sem["trend_type"]).codes
sem["cat_enc"]      = pd.Categorical(sem["categoria"]).codes
sem["tienda_enc"]   = pd.Categorical(sem["id_tienda"]).codes
sem["producto_enc"] = pd.Categorical(sem["id_producto"]).codes

FEATURES = [
    "lag_1w", "lag_2w", "lag_3w", "lag_4w",
    "rolling_mean_2w", "rolling_mean_4w",
    "rolling_std_2w",  "rolling_std_4w",
    "cv", "delta_lag1_lag2",
    "ratio_tienda_vs_media",
    "semana_num", "proporcion_finde",
    "costo_unitario", "precio_venta",
    "costo_almacenamiento_semanal",
    "margen_unitario", "margen_pct",
    "tamaño_m2",
    "trend_enc", "cat_enc",
    "tienda_enc", "producto_enc",
]

print(f"  Semanas totales            : {len(semanas_ord)}")
print(f"  Semanas para entrenamiento : {SEMANA_TEST_INICIO - 1}")
print(f"  Semanas para validación    : {len(semanas_ord) - SEMANA_TEST_INICIO + 1}")
print(f"  Features generadas         : {len(FEATURES)}")

# Dividir train / test
sem_train = sem[sem["semana_num"] < SEMANA_TEST_INICIO].dropna(subset=FEATURES)
sem_test  = sem[sem["semana_num"] >= SEMANA_TEST_INICIO].dropna(subset=FEATURES)

X_train, y_train = sem_train[FEATURES], sem_train["demanda_semanal"]
X_test,  y_test  = sem_test[FEATURES],  sem_test["demanda_semanal"]

print(f"  Filas de entrenamiento     : {len(X_train):,}")
print(f"  Filas de validación        : {len(X_test):,}")



In [ ]:
import mlflow
import mlflow.lightgbm
from mlflow.models.signature import infer_signature

# ── Configuración MLflow ──────────────────────────────────────────────────────
mlflow.set_tracking_uri("file:./mlruns")        # guarda experimentos en local
mlflow.set_experiment("abastecimiento_retail")  # nombre del experimento

SEED = 42

LGBM_PARAMS = {
    "n_estimators":      600,
    "learning_rate":     0.04,
    "num_leaves":        31,
    "min_child_samples": 8,
    "subsample":         0.80,
    "colsample_bytree":  0.80,
    "reg_alpha":         0.1,
    "reg_lambda":        0.2,
    "random_state":      SEED,
    "verbose":          -1,
}

modelos      = {}
pred_test    = {}
pred_prox    = {}
run_ids      = {}   # guardamos los run_id para luego cargar en FastAPI

ultima_semana = sem[sem["semana_num"] == sem["semana_num"].max()].copy()

for nombre_q, alpha in QUANTILES.items():

    with mlflow.start_run(run_name=f"lgbm_{nombre_q}") as run:

        # — Parámetros —
        params = {**LGBM_PARAMS, "objective": "quantile", "alpha": alpha}
        mlflow.log_params(params)
        mlflow.log_param("quantile_name", nombre_q)
        mlflow.log_param("semana_test_inicio", SEMANA_TEST_INICIO)
        mlflow.log_param("n_features", len(FEATURES))
        mlflow.log_param("n_train_rows", len(X_train))

        # — Entrenamiento —
        modelo = lgb.LGBMRegressor(**params)
        modelo.fit(
            X_train, y_train,
            eval_set=[(X_train, y_train)],
            callbacks=[lgb.log_evaluation(period=-1)],
        )
        modelos[nombre_q] = modelo

        # — Predicciones hold-out —
        preds = np.maximum(0, modelo.predict(X_test))
        pred_test[nombre_q] = preds

        # — Métricas (solo para P50 calculamos MAE/MAPE, para P10/P90 pinball) —
        pinball = np.where(
            y_test.values >= preds,
            alpha       * (y_test.values - preds),
            (1 - alpha) * (preds - y_test.values)
        ).mean()
        mlflow.log_metric("pinball_loss_holdout", round(float(pinball), 4))

        if nombre_q == "P50":
            mae  = mean_absolute_error(y_test.values, preds)
            rmse = mean_squared_error(y_test.values, preds) ** 0.5
            mape = (np.abs(y_test.values - preds) / (y_test.values + 1e-9)).mean() * 100
            bias = (preds - y_test.values).mean()
            mlflow.log_metric("mae",  round(mae,  4))
            mlflow.log_metric("rmse", round(rmse, 4))
            mlflow.log_metric("mape", round(mape, 4))
            mlflow.log_metric("bias", round(bias, 4))

        # — Cobertura IC (solo cuando tenemos los 3 modelos: se loguea al final) —

        # — Artefactos: importancia de features —
        imp_df = pd.DataFrame({
            "feature":     FEATURES,
            "importancia": modelo.feature_importances_,
        }).sort_values("importancia", ascending=False)
        imp_path = f"feature_importance_{nombre_q}.csv"
        imp_df.to_csv(imp_path, index=False)
        mlflow.log_artifact(imp_path)

        # — Registro del modelo con firma —
        signature = infer_signature(X_train, modelo.predict(X_train))
        mlflow.lightgbm.log_model(
            lgb_model   = modelo,
            artifact_path = f"model_{nombre_q}",
            signature   = signature,
            input_example = X_train.iloc[:3],
            registered_model_name = f"lgbm_quantile_{nombre_q.lower()}",
        )

        run_ids[nombre_q] = run.info.run_id
        print(f"  Modelo {nombre_q} | run_id: {run.info.run_id} | pinball: {pinball:.4f}")

        # — Predicción semana 14 —
        feat_prox = ultima_semana[FEATURES].copy()
        pred_prox[nombre_q] = np.maximum(0, modelo.predict(feat_prox))

# — Loguear cobertura IC en un run de resumen —
dentro_ic = np.mean(
    (pred_test["P10"] <= y_test.values) & (y_test.values <= pred_test["P90"])
) * 100
print(f"\n  Cobertura IC 80%: {dentro_ic:.1f}%")

# Guardar run_ids para que FastAPI los use
pd.DataFrame([run_ids]).to_json("mlflow_run_ids.json", orient="records")
print("  → mlflow_run_ids.json guardado")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECCIÓN 5 — VALIDACIÓN EN HOLD-OUT (semanas 12-13)
# ─────────────────────────────────────────────────────────────────────────────
print("\n[5/7] VALIDACIÓN EN HOLD-OUT")
print("-" * 50)

y_true = y_test.values
y_pred = pred_test["P50"]

mae  = mean_absolute_error(y_true, y_pred)
rmse = mean_squared_error(y_true, y_pred) ** 0.5
mape = (np.abs(y_true - y_pred) / (y_true + 1e-9)).mean() * 100
bias = (y_pred - y_true).mean()

# Cobertura real del intervalo P10-P90
dentro_ic = np.mean((pred_test["P10"] <= y_true) & (y_true <= pred_test["P90"])) * 100

print(f"  MAE  (P50 vs. real)   : {mae:.2f} unidades/semana")
print(f"  RMSE (P50 vs. real)   : {rmse:.2f} unidades/semana")
print(f"  MAPE (P50 vs. real)   : {mape:.1f}%")
print(f"  Bias (P50 - real)     : {bias:+.2f} unidades (+ = sobreestima)")
print(f"  Cobertura IC 80%      : {dentro_ic:.1f}% (esperado: ~80%)")

# Importancia de features
importancias = pd.DataFrame({
    "feature":     FEATURES,
    "importancia": modelos["P50"].feature_importances_,
}).sort_values("importancia", ascending=False)

# ── Figura de Validación ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Panel 1: Real vs. Predicho (P50)
lim = max(y_true.max(), y_pred.max()) * 1.05
axes[0].scatter(y_true, y_pred, alpha=0.4, color=COLORES["azul_medio"],
                s=25, edgecolors="none")
axes[0].plot([0, lim], [0, lim], "k--", lw=1.5, label="Perfecto")
axes[0].set_xlim(0, lim); axes[0].set_ylim(0, lim)
axes[0].set_title(f"Real vs. Predicho (P50)\nMAPE={mape:.1f}% | MAE={mae:.1f} u",
                  fontsize=12, fontweight="bold")
axes[0].set_xlabel("Demanda real (unidades/semana)")
axes[0].set_ylabel("Pronóstico P50")
axes[0].legend()

# Panel 2: Serie temporal para un SKU ejemplo (STORE_01 / PROD_005)
ej_tienda  = "STORE_01"
ej_prod    = "PROD_005"
serie_ej = sem[(sem["id_tienda"]==ej_tienda) & (sem["id_producto"]==ej_prod)].sort_values("semana_num")
test_ej  = sem_test[(sem_test["id_tienda"]==ej_tienda) & (sem_test["id_producto"]==ej_prod)]
idx_test_ej = test_ej.index

axes[1].plot(serie_ej["semana_num"], serie_ej["demanda_semanal"],
             marker="o", ms=5, color=COLORES["azul_oscuro"], lw=2, label="Real histórico")

if len(test_ej) > 0:
    t_ej = test_ej["semana_num"].values
    axes[1].scatter(t_ej, pred_test["P50"][test_ej.index - X_test.index[0]],
                    color=COLORES["rojo"], zorder=5, s=70, label="Pronóstico P50")
    axes[1].fill_between(
        t_ej,
        pred_test["P10"][test_ej.index - X_test.index[0]],
        pred_test["P90"][test_ej.index - X_test.index[0]],
        color=COLORES["rojo"], alpha=0.2, label="IC 80% (P10–P90)"
    )
    axes[1].axvline(SEMANA_TEST_INICIO - 0.5, color="gray", ls="--", lw=1.5, label="Inicio hold-out")

axes[1].set_title(f"Serie semanal — {ej_tienda} / {ej_prod}\n(Búnuelo, Alimentos)",
                  fontsize=12, fontweight="bold")
axes[1].set_xlabel("Semana")
axes[1].set_ylabel("Unidades/semana")
axes[1].legend(fontsize=9)

# Panel 3: Top features
top_feat = importancias.head(12)
axes[2].barh(top_feat["feature"][::-1], top_feat["importancia"][::-1],
             color=COLORES["azul_medio"], edgecolor="white")
axes[2].set_title("Top 12 features más importantes\n(LightGBM P50, Gain)",
                  fontsize=12, fontweight="bold")
axes[2].set_xlabel("Importancia (Gain)")

plt.suptitle("Validación del Modelo de Forecasting — Hold-out semanas 12-13",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/03_validacion_modelo.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"  → 03_validacion_modelo.png guardado")



In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECCIÓN 6 — OPTIMIZACIÓN NEWSVENDOR
# ─────────────────────────────────────────────────────────────────────────────
print("\n[6/7] OPTIMIZACIÓN NEWSVENDOR")
print("-" * 50)

# ─────────────────────────────────────────────────────────────────────────────
# TEORÍA: Modelo Newsvendor con Incertidumbre Cuantificada
# ─────────────────────────────────────────────────────────────────────────────
#
# Función de costo total esperado:
#   E[CT(Q)] = Cu · E[max(D - Q, 0)]   ← costo de stockout
#            + Co · E[max(Q - D, 0)]   ← costo de overstock
#
# Definiciones de costos:
#   Cu = precio_venta - costo_unitario      (margen perdido por unidad no vendida)
#   Co = costo_almacenamiento_semanal       (costo de guardar 1 unidad 1 semana)
#
# Critical Ratio (CR):
#   CR = Cu / (Cu + Co)   ∈ (0, 1)
#   → CR alto: el costo de NO tener stock supera al costo de sobrestock
#              → conviene pedir MÁS (perfil agresivo)
#   → CR bajo: el costo de guardar exceso supera al costo de perder ventas
#              → conviene pedir MENOS (perfil conservador)
#
# Demanda modelada como Normal(μ, σ²):
#   μ  = pronóstico P50 del modelo
#   σ̂ = (P90 - P10) / (2 × z₀.₉₀)  donde z₀.₉₀ = 1.2816
#
# Cantidad óptima:
#   Q* = μ + z(CR) × σ̂    [donde z(CR) = norm.ppf(CR)]
#
# Pedido final (ajustado por stock disponible):
#   Pedido = max(0, Q* - stock_actual)
#
# RELACIÓN INCERTIDUMBRE → AGRESIVIDAD:
#   - Si σ̂ es ALTA (modelo impreciso) y CR > 0.5 → Q* sube más → pedir más buffer
#   - Si σ̂ es BAJA (modelo preciso)   → Q* ≈ μ   → pedir solo el pronóstico central
# ─────────────────────────────────────────────────────────────────────────────

def newsvendor(mu, sigma, cu, co):
    """Calcula Q* óptimo del modelo Newsvendor."""
    sigma = max(float(sigma), 1e-6)
    cu    = max(float(cu), 0.0)
    co    = max(float(co), 1e-6)
    cr    = cu / (cu + co)
    z     = norm.ppf(cr)
    q_opt = float(mu) + z * sigma

    if cr >= 0.90: agresividad = "MUY AGRESIVO"
    elif cr >= 0.75: agresividad = "AGRESIVO"
    elif cr >= 0.55: agresividad = "MODERADO"
    elif cr >= 0.35: agresividad = "CONSERVADOR"
    else:            agresividad = "MUY CONSERVADOR"

    return {"cr": cr, "z": z, "q_opt": max(0.0, q_opt), "agresividad": agresividad}


def costo_esperado_total(Q, mu, sigma, cu, co):
    """E[CT(Q)] usando la función de pérdida Normal."""
    sigma = max(float(sigma), 1e-6)
    z     = (float(Q) - float(mu)) / sigma
    phi   = norm.pdf(z)
    Phi   = norm.cdf(z)
    stockout  = sigma * phi - (Q - mu) * (1 - Phi)
    overstock = (Q - mu) * Phi + sigma * phi
    return cu * max(0, stockout) + co * max(0, overstock)


# Construir tabla de resultados para la semana 14
ultima_semana = ultima_semana.copy()
ultima_semana["pred_P10"] = pred_prox["P10"]
ultima_semana["pred_P50"] = pred_prox["P50"]
ultima_semana["pred_P90"] = pred_prox["P90"]
ultima_semana["sigma_hat"] = (ultima_semana["pred_P90"] - ultima_semana["pred_P10"]) / (2 * 1.2816)

resultados = []
for _, row in ultima_semana.iterrows():
    mu      = float(row["pred_P50"])
    sigma   = float(row["sigma_hat"])
    cu      = float(row["margen_unitario"])
    co      = float(row["costo_almacenamiento_semanal"])

    # Stock actual
    stock_row = stock_df[
        (stock_df["id_tienda"] == row["id_tienda"]) &
        (stock_df["id_producto"] == row["id_producto"])
    ]
    stock = int(stock_row["stock_actual"].values[0]) if len(stock_row) else 0

    nv = newsvendor(mu, sigma, cu, co)

    q_star = nv["q_opt"]
    pedido = max(0.0, q_star - stock)

    # Costo esperado: política newsvendor vs. política naive (pedir solo μ)
    ct_opt   = costo_esperado_total(q_star, mu, sigma, cu, co)
    ct_naive = costo_esperado_total(mu,     mu, sigma, cu, co)
    ahorro   = ct_naive - ct_opt

    # Nivel de servicio implícito
    ns = norm.cdf((q_star - mu) / max(sigma, 1e-6)) * 100

    resultados.append({
        "id_tienda":          row["id_tienda"],
        "id_producto":        row["id_producto"],
        "nombre_producto":    row["nombre"],
        "categoria":          row["categoria"],
        "ciudad":             row["ciudad"],
        "trend_type":         row["trend_type"],
        "P10":                round(row["pred_P10"], 1),
        "P50_pronostico":     round(mu, 1),
        "P90":                round(row["pred_P90"], 1),
        "sigma_hat":          round(sigma, 2),
        "margen_unitario_Cu": cu,
        "costo_hold_Co":      co,
        "critical_ratio_CR":  round(nv["cr"], 4),
        "z_score":            round(nv["z"], 3),
        "Q_optimo":           round(q_star, 1),
        "stock_actual":       stock,
        "PEDIDO_RECOMENDADO": int(round(max(0, pedido))),
        "nivel_servicio_pct": round(ns, 1),
        "agresividad":        nv["agresividad"],
        "costo_esperado_opt": round(ct_opt, 2),
        "costo_esperado_naive": round(ct_naive, 2),
        "ahorro_vs_naive_COP": round(ahorro, 2),
    })

resultados_df = pd.DataFrame(resultados)
resultados_df.to_csv(f"{OUTPUT_DIR}/04_tabla_pedidos_semana14.csv", index=False)

print(f"  SKU-Tiendas procesados   : {len(resultados_df)}")
print(f"  Pedido total recomendado : {resultados_df['PEDIDO_RECOMENDADO'].sum():,} unidades")
print(f"  Ahorro vs. política naive: COP {resultados_df['ahorro_vs_naive_COP'].sum():,.0f}")
print(f"  Nivel de servicio prom.  : {resultados_df['nivel_servicio_pct'].mean():.1f}%")
print(f"  CR promedio              : {resultados_df['critical_ratio_CR'].mean():.3f}")
print(f"\n  Distribución de perfil de pedido:")
print(resultados_df["agresividad"].str.split(" ").str[1].value_counts().to_string())

# Vista resumen por producto
resumen_prod = (
    resultados_df.groupby(["id_producto","nombre_producto","categoria"])
    .agg(
        CR_prom     = ("critical_ratio_CR", "mean"),
        NS_prom     = ("nivel_servicio_pct", "mean"),
        pedido_total= ("PEDIDO_RECOMENDADO", "sum"),
        ahorro_total= ("ahorro_vs_naive_COP", "sum"),
    )
    .reset_index()
    .sort_values("CR_prom", ascending=False)
)
print(f"\n  Resumen por producto (todas las tiendas):")
print(resumen_prod.to_string(index=False))

print("Cantidad óptima por producto en cada tienda (Q_optimo y PEDIDO_RECOMENDADO):")
display(resultados_df[['id_tienda', 'id_producto', 'nombre_producto', 'Q_optimo', 'stock_actual', 'PEDIDO_RECOMENDADO']])

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECCIÓN 7 — VISUALIZACIÓN OPTIMIZACIÓN + RESUMEN EJECUTIVO
# ─────────────────────────────────────────────────────────────────────────────
print("\n[7/7] VISUALIZACIÓN FINAL + RESUMEN EJECUTIVO")
print("-" * 50)

fig = plt.figure(figsize=(22, 18))
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.50, wspace=0.40)

# Re-calculate patron_dow and incremento for the executive summary
# These were previously calculated in Section 2 (EDA)
patron_dow = df.groupby("dia_semana")["unidades_vendidas"].mean()
incremento = (patron_dow.iloc[4:].mean() / patron_dow.iloc[:4].mean() - 1) * 100

# ── Panel 1: Curva de Costo Esperado vs. Q (producto con CR más alto) ─────────
ax1 = fig.add_subplot(gs[0, :2])
ej_row = resultados_df.sort_values("critical_ratio_CR", ascending=False).iloc[0]
mu_e  = ej_row["P50_pronostico"]
sig_e = max(ej_row["sigma_hat"], 1.0)
cu_e  = ej_row["margen_unitario_Cu"]
co_e  = ej_row["costo_hold_Co"]
q_e   = ej_row["Q_optimo"]

Q_rng  = np.linspace(max(0, mu_e - 4*sig_e), mu_e + 4*sig_e, 300)
ct_tot = [costo_esperado_total(q, mu_e, sig_e, cu_e, co_e) for q in Q_rng]
ct_su  = [cu_e * max(0, norm.expect(lambda d: max(d-q, 0),
           loc=mu_e, scale=sig_e)) for q in Q_rng]
ct_ov  = [co_e * max(0, norm.expect(lambda d: max(q-d, 0),
           loc=mu_e, scale=sig_e)) for q in Q_rng]

ax1.plot(Q_rng, ct_tot, color=COLORES["azul_oscuro"], lw=2.8, label="E[CT(Q)] — Costo total")
ax1.plot(Q_rng, ct_su,  color=COLORES["rojo"],        lw=1.8, ls="--", label="Costo stockout")
ax1.plot(Q_rng, ct_ov,  color=COLORES["verde"],       lw=1.8, ls="--", label="Costo overstock")
ax1.axvline(q_e,   color=COLORES["rojo"],    lw=2,  ls="-.", label=f"Q* = {q_e:.0f} u")
ax1.axvline(mu_e,  color=COLORES["gris"],    lw=1.5, ls=":",  label=f"P50 = {mu_e:.0f} u")
ax1.fill_betweenx(
    [0, max(ct_tot)*1.05],
    ej_row["P10"], ej_row["P90"],
    alpha=0.08, color=COLORES["rojo"], label=f"IC 80% demanda"
)
ax1.set_ylim(0, max(ct_tot) * 1.15)
ax1.set_title(
    f"Curva de Costo Esperado vs. Cantidad Pedida\n"
    f"{ej_row['nombre_producto']} | {ej_row['id_tienda']} — {ej_row['ciudad']} | "
    f"CR={ej_row['critical_ratio_CR']:.2f} | {ej_row['agresividad']}",
    fontsize=11, fontweight="bold"
)
ax1.set_xlabel("Cantidad ordenada Q (unidades/semana)")
ax1.set_ylabel("Costo esperado (COP)")
ax1.legend(fontsize=9)

# ── Panel 2: Distribución de Critical Ratios por categoría ────────────────────
ax2 = fig.add_subplot(gs[0, 2])
for cat, color in [("Bebidas", COLORES["azul_oscuro"]), ("Alimentos", COLORES["naranja"])]:
    sub = resultados_df[resultados_df["categoria"] == cat]["critical_ratio_CR"]
    ax2.hist(sub, bins=12, alpha=0.65, color=color, edgecolor="white", label=cat)
ax2.axvline(0.5, color="gray", ls="--", lw=1.5)
ax2.set_title("Distribución de Critical\nRatios por Categoría", fontsize=11, fontweight="bold")
ax2.set_xlabel("CR = Cu / (Cu + Co)")
ax2.set_ylabel("Nro SKU-Tiendas")
ax2.legend()

# ── Panel 3: Q* vs. Pedido final por producto (top/bottom CR) ─────────────────
ax3 = fig.add_subplot(gs[1, :2])
df_vis = resultados_df.sort_values("critical_ratio_CR", ascending=False).head(30)
x_pos  = range(len(df_vis))
ax3.bar(x_pos, df_vis["Q_optimo"],            color=COLORES["azul_claro"],  label="Q* óptimo")
ax3.bar(x_pos, df_vis["PEDIDO_RECOMENDADO"],   color=COLORES["azul_oscuro"], alpha=0.85,
        label="Pedido final (Q* - stock)")
ax3.bar(x_pos, df_vis["stock_actual"],         color=COLORES["rojo"],        alpha=0.45,
        label="Stock actual")
etiq = [f"{r['id_tienda'][-2:]}\n{r['nombre_producto'][:7]}"
        for _, r in df_vis.iterrows()]
ax3.set_xticks(list(x_pos))
ax3.set_xticklabels(etiq, fontsize=6.5, rotation=45, ha="right")
ax3.set_title("Q*, Pedido y Stock — Top 30 SKU-Tiendas por Critical Ratio",
              fontsize=11, fontweight="bold")
ax3.set_ylabel("Unidades")
ax3.legend(fontsize=9)

# ── Panel 4: Nivel de Servicio vs. CR (scatter) ───────────────────────────────
ax4 = fig.add_subplot(gs[1, 2])
colores_sc = [COLORES["azul_oscuro"] if c=="Bebidas" else COLORES["naranja"]
              for c in resultados_df["categoria"]]
sc = ax4.scatter(
    resultados_df["critical_ratio_CR"],
    resultados_df["nivel_servicio_pct"],
    c=resultados_df["sigma_hat"],
    cmap="RdYlGn_r", s=55, edgecolors="none", alpha=0.75
)
plt.colorbar(sc, ax=ax4, label="σ̂ (incertidumbre)")
ax4.axhline(90, color=COLORES["rojo"], ls="--", lw=1.2, label="Meta NS 90%")
ax4.set_xlabel("Critical Ratio (CR)")
ax4.set_ylabel("Nivel de Servicio (%)")
ax4.set_title("CR vs. Nivel de Servicio\n(color = incertidumbre σ̂)", fontsize=11, fontweight="bold")
ax4.legend(fontsize=9)

# ── Panel 5: Ahorro por producto (todas las tiendas) ─────────────────────────
ax5 = fig.add_subplot(gs[2, :2])
ahorro_prod = (
    resultados_df.groupby(["nombre_producto","categoria"])["ahorro_vs_naive_COP"]
    .sum().reset_index().sort_values("ahorro_vs_naive_COP")
)
colores_ah = [COLORES["naranja"] if c=="Alimentos" else COLORES["azul_oscuro"]
              for c in ahorro_prod["categoria"]]
bars_ah = ax5.barh(ahorro_prod["nombre_producto"], ahorro_prod["ahorro_vs_naive_COP"],
                   color=colores_ah, edgecolor="white")
ax5.axvline(0, color="gray", lw=1.2)
for bar in bars_ah:
    ax5.text(bar.get_width() + abs(bar.get_width())*0.02,
             bar.get_y() + bar.get_height()/2,
             f"COP {bar.get_width():,.0f}", va="center", fontsize=8.5)
ax5.set_title("Ahorro esperado total por Producto\n(Newsvendor vs. Política Naive Q=μ)",
              fontsize=11, fontweight="bold")
ax5.set_xlabel("Ahorro COP (suma 20 tiendas)")

# ── Panel 6: σ̂ vs. Pedido por trend_type ─────────────────────────────────────
ax6 = fig.add_subplot(gs[2, 2])
trend_colors = {"up": COLORES["verde"], "down": COLORES["rojo"],
                "seasonal": COLORES["naranja"], "random": COLORES["gris"]}
for tt, color in trend_colors.items():
    sub = resultados_df[resultados_df["trend_type"] == tt]
    ax6.scatter(sub["sigma_hat"], sub["PEDIDO_RECOMENDADO"],
                c=color, s=50, alpha=0.7, label=tt, edgecolors="none")
ax6.set_xlabel("Incertidumbre σ̂ (unidades/semana)")
ax6.set_ylabel("Pedido recomendado (unidades)")
ax6.set_title("Incertidumbre σ̂ vs. Pedido\npor Tipo de Tendencia", fontsize=11, fontweight="bold")
ax6.legend(title="trend_type", fontsize=9)

fig.suptitle(
    "Optimización Newsvendor — Pedido Óptimo por SKU-Tienda | Semana 14",
    fontsize=15, fontweight="bold", y=1.01
)
plt.savefig(f"{OUTPUT_DIR}/04_optimizacion_newsvendor.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"  → 04_optimizacion_newsvendor.png guardado")

# ── PAYLOAD REAL PARA PROBAR LA API ──────────────────────────────────────────
sku_ejemplo = ultima_semana.iloc[0]

payload = {}
for col in FEATURES:
    val = sku_ejemplo[col]
    if col == "semana_num":
        payload[col] = int(val)
    elif col in ["trend_enc", "cat_enc", "tienda_enc", "producto_enc"]:
        payload[col] = int(val)
    else:
        payload[col] = float(val)

payload["stock_actual"] = int(stock_df[
    (stock_df["id_tienda"]   == sku_ejemplo["id_tienda"]) &
    (stock_df["id_producto"] == sku_ejemplo["id_producto"])
]["stock_actual"].values[0])

import json
print("SKU:", sku_ejemplo["id_tienda"], "|", sku_ejemplo["nombre"])
print("margen_unitario          :", payload["margen_unitario"])
print("costo_almacenamiento_semanal:", payload["costo_almacenamiento_semanal"])
print()
print("── PAYLOAD COMPLETO ──")
print(json.dumps(payload, indent=2))
